<h1><b>Utilizando recursos da biblioteca Scikit-Learn para avaliar modelos de Machine Learning</b></h1>
<h3>Prof. Dr. Rafael Rieder</h3>
<h4>Exemplo para aulas relacionadas ao Aprendizado de M?quina</h4>
<h5>Universidade de Passo Fundo. ?ltima atualiza??o: 27/10/2023</h5>

In [ ]:
# Model Evaluation (Regression and Classification)
# Credits: Angelica Lo Duca
# Source: https://towardsdatascience.com/model-evaluation-in-scikit-learn-abce32ee4a99

# Changed to use as a learning resource in ML class: Rafael Rieder
# University of Passo Fundo, 2023

# Dataset: https://archive.ics.uci.edu/ml/datasets/wine+quality
# Sources:
# Dua, D. and Graff, C. (2019). UCI Machine Learning Repository [http://archive.ics.uci.edu/ml]. Irvine, CA: the University of California, School of Information and Computer Science.
# P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis. Modeling wine preferences by data mining from physicochemical properties. In Decision Support Systems, Elsevier, 47(4):547-553, 2009.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, roc_auc_score, fbeta_score
from sklearn.metrics import average_precision_score, precision_recall_curve, PrecisionRecallDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

Leitura dos dois arquivos de dados (vinhos tinto e branco), e concatenando-os em um s? dataset. Ao final, conferindo o n?mero de colunas e a sa?da exibindo os primeiros e ?ltimos 5 registros.

In [ ]:
targets = ['red', 'white']
df_list = []
df = pd.DataFrame()
mypath = f""
for target in targets:
    df_temp = pd.read_csv(mypath + f"winequality-{target}.csv", sep=';')
    df_temp['target'] = target
    df_list.append(df_temp)
    print(df_temp.shape)
df = pd.concat([df_list[0], df_list[1]])

print(df.shape)

df.head(5).append(df.tail(5))

Fun??o para codificar e transformar dados categ?ricos em num?ricos. Para tanto, trabalhamos apenas com colunas do tipo "object" (e evitar em mexer em dados que j? s?o num?ricos).

In [ ]:
def transform_categorical(data):
    categories = (data.dtypes =="object")
    cat_cols = list(categories[categories].index)
    label_encoder = LabelEncoder()
    for col in cat_cols:
        data[col] = label_encoder.fit_transform(data[col])

Fun??o para dimensionar colunas num?ricas (transformar as caracter?sticas dos dados, para deix?-las na mesma escala em um determinado intervalo - no caso, entre 0 e 1).

In [ ]:
def scale_numerical(data):
    scaler = MinMaxScaler()
    data[data.columns] = scaler.fit_transform(data[data.columns])

<h2><b>Avaliando modelos de regress?o</b></h2>
<p>M?tricas a serem aplicadas:</p>


*   Erro m?dio absoluto (MAE): a m?dia da diferen?a entre o valor real e o previsto. Ele mede a dist?ncia entre as previs?es e a sa?da real. Quanto menor o MAE, melhor o modelo;
*   Raiz quadrada do erro m?dio (RMSE): a raiz quadrada do erro quadr?tico m?dio (MSE) calcula a m?dia do quadrado da diferen?a entre os valores reais e os previstos. Usada para expressar a acur?cia dos resultados num?ricos;
*   R? score (coeficiente de determina??o): propor??o de vari?ncia em Y que pode ser explicada por X. Mostra quanto o modelo se ajusta ? amostra (quanto maior o R?, melhor o ajuste).



<u>Tarefa: prever o pH de cada registro, considerando as demais caracter?sticas.</u>
Come?amos retirando a coluna de pH do conjunto, salvando essa estrutura em X.
Em seguida, salvamos s? a coluna pH em Y.

In [ ]:
X = df.drop("pH", axis = 1)
y = df["pH"]

Transformando categorias em n?meros, e normalizando n?meros na escala 0-1:

In [ ]:
transform_categorical(X)
scale_numerical(X)
X.head(5).append(X.tail(5))

Dividindo os datasets X e Y em conjuntos de treino e teste.
Definiu-se o tamanho do conjunto de teste em 20% (logo, treino ser? 80%). J? "random_state" ? usado para reproduzir o problema da mesma forma toda vez que ? executado (controla o "embaralhamento").

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

Fun??o para rodar o modelo (fit), realizar as predi??es (predict) e calcular os scores de desempenho) para t?cnicas de Regress?o aplicadas.

Construindo um modelo de Regress?o Linear, e testando seu desempenho na predi??o de resultados:

In [ ]:
def run_experiment_regression(model, X, y):
   
    # Ajustar o modelo aos dados de treinamento
    model.fit(X_train, y_train)
    
    # Fazer previs?es nos dados de teste
    y_pred = model.predict(X_test)
    
    # Calcular m?tricas de desempenho
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Retornar as m?tricas de desempenho
    return {
        "R Mean Squared Error (MSE)": rmse,
        "Mean Absolute Error (MAE)": mae,
        "R-squared (R2)": r2
    }

# Exemplo de uso com uma regress?o linear
from sklearn.linear_model import LinearRegression

# Criar um modelo de regress?o linear
modelo_regressao_linear = LinearRegression()

# Executar o experimento
resultados = run_experiment_regression(modelo_regressao_linear, X, y)

# Imprimir os resultados
print("Resultados do experimento de regress?o:")
for metric, valor in resultados.items():
    print(f"{metric}: {valor}")
    # inserir c?digo aqui

Construindo um modelo de Gradiente Descendente Estoc?stico, e testando seu desempenho na predi??o de resultados:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Criar um modelo de Gradiente Descendente Estoc?stico para regress?o
modelo_sgd = SGDRegressor(random_state=42)

# Ajustar o modelo aos dados de treinamento
modelo_sgd.fit(X_train, y_train)

# Fazer previs?es nos dados de teste
y_pred = modelo_sgd.predict(X_test)

# Calcular m?tricas de desempenho
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Imprimir os resultados
print("M?tricas de desempenho do modelo de Gradiente Descendente Estoc?stico:")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")

Construindo um modelo de regress?o Random Forest, e testando seu desempenho na predi??o de resultados:

In [ ]:
# Dividir os dados em conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Criar um modelo de regress?o Random Forest
modelo_random_forest = RandomForestRegressor(random_state=42)

# Ajustar o modelo aos dados de treinamento
modelo_random_forest.fit(X_train, y_train)

# Fazer previs?es nos dados de teste
y_pred = modelo_random_forest.predict(X_test)

# Calcular m?tricas de desempenho
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Imprimir os resultados
print("M?tricas de desempenho do modelo de regress?o Random Forest:")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")

<h2><b>Avaliando modelos de classifica??o</b></h2>
<p>M?tricas a serem aplicadas: </p>

*   Matriz de confus?o: gr?fico para verificar verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos. Indica os erros e acertos do seu modelo, comparando com o resultado esperado;
*   Acur?cia: medida intuitiva que indica uma performance geral do modelo. Dentre todas as classifica??es, quantas o modelo classificou corretamente;
*   Precis?o: dentre todas as classifica??es de classe Positivo que o modelo fez, mostra quantas est?o corretas. A precis?o pode ser usada em uma situa??o em que os Falsos Positivos s?o considerados mais prejudiciais que os Falsos Negativos;
*   Revoca??o/ Sensibilidade (Recall): dentre todas as situa??es de classe Positivo como valor esperado, quantas est?o corretas. Pode ser usada em uma situa??o em que os Falsos Negativos s?o considerados mais prejudiciais que os Falsos Positivos;
*   F1-Score: m?dia harm?nica entre precis?o e recall. Ele ser? alto quando ambas as m?tricas base forem altas e similares, ou seja, ? maior quando h? um "meio termo" entre precis?o e recall;
*   Curva ROC: mostra a rela??o entre a taxa de verdadeiros positivos (true positive rate ? TPR) e a taxa de falsos positivos (false positive rate ? FPR) para diferentes thresholds. Se a curva ROC de um modelo se aproxima bastante do ponto (0.0, 1.0), na parte alta e esquerda do gr?fico, esse modelo ? bom;
*   ROC AUC Score: para medir a proximidade com o ponto (0.0, 1.0) utiliza-se a ?rea sob a curva ROC (area under the curve ? AUC). Quanto mais pr?ximo de 1.0 ? o score da ?rea sob a curva ROC, melhor ? o modelo;
*   Curva Precision-Recall: plota a rela??o entre precis?o e revoca??o ? medida que o limite de decis?o ? alterado. Uma boa curva PR tem maior AUC (?rea sob a curva). A precis?o ajuda a destacar a relev?ncia dos resultados de revoca??o (quanto mais pr?ximo das partes alta e direita do gr?fico, melhor);
*   Average Precision (AP): ?rea debaixo da curva de precision X recall. ?til para comparar diferentes modelos de detec??o de objetos.

<u>Tarefa: prever o tipo de vinho (tinto ou branco).</u>
Come?amos retirando a coluna "target" do conjunto, salvando essa estrutura em X.
Em seguida, salvamos s? a coluna "target" em Y.

In [ ]:
X = df.drop("target", axis = 1)
y = df["target"]

Transformando categorias em n?meros, e normalizando n?meros na escala 0-1:

In [ ]:
transform_categorical(X)
scale_numerical(X)

Transformar tamb?m as classes de vinho (0 e 1):

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

Dividindo os datasets X e Y em conjuntos de treino e teste.
Definiu-se o tamanho do conjunto de teste em 20% (logo, treino ser? 80%). J? "random_state" ? usado para reproduzir o problema da mesma forma toda vez que ? executado (controla o "embaralhamento").

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

Fun??o para rodar o modelo (fit), realizar as predi??es (predict) e calcular os scores de desempenho) para t?cnicas de Classifica??o aplicadas.

In [ ]:
def run_experiment_classification(model, name):
    # inserir c?digo aqui

Construindo um modelo de Random Forest, e testando seu desempenho na predi??o de resultados:

In [ ]:
modelo_random_forest = RandomForestRegressor(random_state=42)

# Ajustar o modelo aos dados de treinamento
modelo_random_forest.fit(X_train, y_train)

# Fazer previs?es nos dados de teste
y_pred = modelo_random_forest.predict(X_test)

# Calcular m?tricas de desempenho
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Imprimir os resultados
print("M?tricas de desempenho do modelo Random Forest para regress?o:")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")

Construindo um modelo de ?rvore de Decis?o, e testando seu desempenho na predi??o de resultados:

In [ ]:
from sklearn.tree import DecisionTreeRegressor
# Criar um modelo de ?rvore de Decis?o para regress?o
modelo_arvore_decisao = DecisionTreeRegressor(random_state=42)

# Ajustar o modelo aos dados de treinamento
modelo_arvore_decisao.fit(X_train, y_train)

# Fazer previs?es nos dados de teste
y_pred = modelo_arvore_decisao.predict(X_test)

# Calcular m?tricas de desempenho
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Imprimir os resultados
print("M?tricas de desempenho do modelo de ?rvore de Decis?o para regress?o:")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")

Construindo um modelo de An?lise Discriminante Linear, e testando seu desempenho na predi??o de resultados:

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
# Criar um modelo de An?lise Discriminante Linear (LDA)
modelo_lda = LinearDiscriminantAnalysis()

# Ajustar o modelo aos dados de treinamento
modelo_lda.fit(X_train, y_train)

# Fazer previs?es nos dados de teste
y_pred = modelo_lda.predict(X_test)

# Calcular a acur?cia do modelo
acuracia = accuracy_score(y_test, y_pred)

# Imprimir a acur?cia e o relat?rio de classifica??o
print(f"Acur?cia do modelo: {acuracia}")
print("Relat?rio de Classifica??o:\n", classification_report(y_test, y_pred))

In [ ]:
def run_experiment_classification(model, name):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap='GnBu')
    _ = disp.ax_.set_title("Matriz de confus??o")
    plt.show()

    print('====================')
    print('Precision....: %.3f' % precision_score(y_test, y_pred))
    print('Recall.......: %.3f' % recall_score(y_test, y_pred))
    print('F1 Score.....: %.3f' % f1_score(y_test, y_pred))
    print('Accuracy.....: %.3f' % accuracy_score(y_test, y_pred))
    print('F-Beta Score.: %.3f' % fbeta_score(y_test, y_pred, average='micro', beta=0.5))
    print('Roc AUC Score: %.3f' % roc_auc_score(y_test, y_pred, average='micro'))
    print('====================')

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    plt.xlabel('Taxa de Falsos Positivos')
    plt.ylabel('Taxa de Verdadeiros Positivos')
    plt.title('Curva ROC')
    plt.plot(fpr, tpr)
    plt.show()

    print('')
    pr, rc, th = precision_recall_curve(y_test, y_pred)
    disp = PrecisionRecallDisplay(precision=pr, recall=rc, estimator_name=name, average_precision=average_precision_score(y_test, y_pred, average='micro'))
    disp.plot()
    _ = disp.ax_.set_title("Curva Precision Recall")
    plt.show()

In [ ]:
run_experiment_classification(modelo_lda, "LDA")